# Use proteomics data from PRIDE Data (adipose tissue)
This notebook shows how `acore` can be used to download data from
the Proteomics Identifications Database - PRIDE -
([ebi.ac.uk/pride/](https://www.ebi.ac.uk/pride/))
and parse the data to be used in the analytics core
and quickly formated to start analyzing them with the functionality in the analytics core.

> based on CKG recipe: [Download PRIDE Data](https://ckg.readthedocs.io/en/latest/notebooks/recipes/Download_PRIDE_data.html)

```{warning}
The outputs shown in this notebook are cached: it is not executed during the
documentation build. To refresh them, re-run the notebook locally and commit the
updated `.ipynb` file.
```

In [1]:
%pip install acore

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

import acore.io

## Parameters
Specify the PRIDE identifier and file to be downloaded
and where to store intermediate files.

In [3]:
pxd_id: str = "PXD008541"  # PRIDE identifier
fname = "SearchEngineResults_secretome.zip.rar"  # file to download
folder_downloads = Path("downloaded")  # folder to download the file
folder_unzipped = Path("unzipped")  # folder to uncompress the file

## Specify the PRIDE identifier and file to be downloaded

We can use functionality in `acore` to directly download data files from EBI's
PRIDE database [ebi.ac.uk/pride/](https://www.ebi.ac.uk/pride/).
For that you just need to specify the
PRIDE identifier for the project (`PXD_...`) and the name of the file to download.
In this case, the project identifier is `PXD008541` and the file we will use
is `SearchEngineResults_secretome.zip.rar`,
a RAR compressed file with the output files from MaxQuant.

In [4]:
ret = acore.io.download_PRIDE_data(pxd_id=pxd_id, file_name=fname, to=folder_downloads)
ret["acore_downloaded_file"] = folder_downloads / fname
ret

File already downloaded, skipping: downloaded/SearchEngineResults_secretome.zip.rar


{'accession': 'PXD008541',
 'title': 'Human primary brown and white fat cell secretome',
 'additionalAttributes': [],
 'projectDescription': 'Secreted proteins from adipose tissue play a role in metabolic cross-talk and homeostasis. We performed high sensitivity mass spectrometry-based proteomics on the cell media of human adipocytes derived from the supraclavicular brown adipose and from the subcutaneous white adipose depots of adult humans. We identified 471 potentially secreted proteins covering interesting protein categories such as hormones, growth factors, extracellular matrix proteins and proteins of the complement system, which were differentially regulated in brown and white adipocytes. A total of 101 proteins were exclusively quantified in brown adipocytes and among these were ependymin-related protein 1 (EPDR1). Functional studies suggested a role for EPDR1 in thermogenic adipogenesis. In conclusion, we report substantial differences between the secretomes of brown and white

## Decompress rar File
Pride results are compressed by the researcher themself, so many different file
formats can be found. Here it was stored as a RAR archive. You will need to have
a system installation of a rar archive tool to decompress the file, find it
via [google](https://www.google.com/search?q=unrar+tool&oq=unrar+tool).

In [5]:
# ! you need a system installation of a rar archive tool
acore.io.unrar(filepath=ret["acore_downloaded_file"], to=folder_unzipped)

The list of files within the compressed folder

In [6]:
list(folder_unzipped.iterdir())

[PosixPath('unzipped/peptides.txt'),
 PosixPath('unzipped/modificationSpecificPeptides.txt'),
 PosixPath('unzipped/experimentalDesignTemplate.txt'),
 PosixPath('unzipped/parameters.txt'),
 PosixPath('unzipped/msms.txt'),
 PosixPath('unzipped/proteinGroups.txt')]

## Read and clean the data
We use the proteinGroups file that contains the proteomics data processed
using MaxQuant software.

In [7]:
fpath_proteinGroups = folder_unzipped / "proteinGroups.txt"
index_cols = [
    "Majority protein IDs",
]
data = pd.read_csv(fpath_proteinGroups, index_col=index_cols, sep="\t")
data.sample(5)

,Protein IDs,Peptide counts (all),Peptide counts (razor+unique),Peptide counts (unique),Protein names,Gene names,Fasta headers,Number of proteins,Peptides,Razor + unique peptides,...,Contaminant,id,Peptide IDs,Peptide is razor,Mod. peptide IDs,Evidence IDs,MS/MS IDs,Best MS/MS,Oxidation (M) site IDs,Oxidation (M) site positions
Majority protein IDs,,,,,,,,,,,,,,,,,,,,,
P62861;E9PR30,P62861;E9PR30;E9PM49,5;4;1,5;4;1,5;4;1,40S ribosomal protein S30,FAU,>sp|P62861|RS30_HUMAN 40S ribosomal protein S3...,3,5,5,...,NaN,"1,395",5322;5323;9204;13738;17323,True;True;True;True;True,5555;5556;9606;14473;18234,69305;69306;69307;69308;69309;69310;69311;6931...,52669;52670;52671;52672;52673;52674;52675;5267...,52670;52681;91908;144343;189525,NaN,NaN
O43556;E9PEH6;O43556-3;C9JR67;G5E9K6;B7Z2R4,O43556;E9PEH6;O43556-3;C9JR67;G5E9K6;B7Z2R4,1;1;1;1;1;1,1;1;1;1;1;1,1;1;1;1;1;1,Epsilon-sarcoglycan,SGCE,>sp|O43556|SGCE_HUMAN Epsilon-sarcoglycan OS=H...,6,1,1,...,NaN,140,12306,True,12977,167325;167326,127147,127147,NaN,NaN
Q9BXK5;F2Z2C3;Q8IZP5;E9PDD6;Q9BXK5-2,Q9BXK5;F2Z2C3;Q8IZP5;E9PDD6;Q9BXK5-2,1;1;1;1;1,1;1;1;1;1,1;1;1;1;1,Bcl-2-like protein 13,BCL2L13,>sp|Q9BXK5|B2L13_HUMAN Bcl-2-like protein 13 O...,5,1,1,...,NaN,363,19158,True,20161,266656;266657,208301,208301,NaN,NaN
P30042;P30042-2;H7C1F6,P30042;P30042-2;H7C1F6;H7C2G3;H7C455;F2Z2Q0;H7...,3;3;3;1;1;1;1,3;3;3;1;1;1;1,3;3;3;1;1;1;1,"ES1 protein homolog, mitochondrial",C21orf33,">sp|P30042|ES1_HUMAN ES1 protein homolog, mito...",7,3,3,...,NaN,548,4559;5804;18347,True;True;True,4758;6055;19311,59287;75176;75177;75178;75179;256625;256626;25...,45103;56670;200616;200617;200618;200619,45103;56670;200617,NaN,NaN
Q8TD19,Q8TD19;G3V459,10;1,10;1,10;1,Serine/threonine-protein kinase Nek9,NEK9,>sp|Q8TD19|NEK9_HUMAN Serine/threonine-protein...,2,10,10,...,NaN,"1,703",610;3028;3906;9816;9837;9913;10665;16047;16838...,True;True;True;True;True;True;True;True;True;True,645;3170;4083;10247;10272;10354;11159;16888;17...,8482;8483;8484;8485;8486;8487;40268;52008;5200...,6897;6898;30856;39772;39773;98882;99605;100462...,6897;30856;39773;98882;99605;100462;109153;174...,NaN,NaN


We mark the protein group by the first protein in the group, ensuring that the protein
group is still unique.

In [8]:
new_index = data.index.str.split(";").str[0].rename("first_prot")
assert new_index.is_unique
data = data.reset_index()
data.index = new_index
data

,Majority protein IDs,Protein IDs,Peptide counts (all),Peptide counts (razor+unique),Peptide counts (unique),Protein names,Gene names,Fasta headers,Number of proteins,Peptides,...,Contaminant,id,Peptide IDs,Peptide is razor,Mod. peptide IDs,Evidence IDs,MS/MS IDs,Best MS/MS,Oxidation (M) site IDs,Oxidation (M) site positions
first_prot,,,,,,,,,,,,,,,,,,,,,
Q13443,Q13443;C9J6H5;C9JPM3;F8WC54;Q13443-2;A0AVL1,Q13443;C9J6H5;C9JPM3;F8WC54;Q13443-2;A0AVL1,1;1;1;1;1;1,1;1;1;1;1;1,1;1;1;1;1;1,Disintegrin and metalloproteinase domain-conta...,ADAM9,>sp|Q13443|ADAM9_HUMAN Disintegrin and metallo...,6,1,...,NaN,0,2726,True,2849,36123;36124;36125;36126;36127;36128;36129;3613...,27484;27485;27486;27487;27488;27489;27490;2749...,27490,NaN,NaN
Q08209,Q08209;Q08209-2;E7ETC2;A1A441;Q08209-3;P16298;...,Q08209;Q08209-2;E7ETC2;A1A441;Q08209-3;P16298;...,4;4;4;4;4;2;2;2;2;2;2;1;1;1;1;1;1;1;1,4;4;4;4;4;2;2;2;2;2;2;1;1;1;1;1;1;1;1,4;4;4;4;4;2;2;2;2;2;2;1;1;1;1;1;1;1;1,Serine/threonine-protein phosphatase 2B cataly...,PPP3CA;PPP3CB,>sp|Q08209|PP2BA_HUMAN Serine/threonine-protei...,19,4,...,NaN,1,699;6176;15271;16451,True;True;True;True,738;6440;16080;17320,9593;9594;9595;9596;80162;80163;80164;80165;80...,7726;7727;60560;60561;60562;163519;163520;1635...,7727;60562;163524;178974,NaN,NaN
A1L4H1,A1L4H1,A1L4H1,1,1,1,Soluble scavenger receptor cysteine-rich domai...,SSC5D,>sp|A1L4H1|SRCRL_HUMAN Soluble scavenger recep...,1,1,...,NaN,2,12914,True,13624,177453;177454,136581;136582,136581,NaN,NaN
P63092,P63092;P04899;P08754;P63096;P09471;P38405;P190...,P63092;P04899;P08754;P63096;P09471;P38405;P190...,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1,Guanine nucleotide-binding protein G(s) subuni...,GNAS;GNAI2;GNAI3;GNAI1;GNAO1;GNAL;GNAT2;GNAT3;...,>sp|P63092|GNAS2_HUMAN Guanine nucleotide-bind...,22,1,...,NaN,3,10218,True,10674,137064;137065;137066;137067;137068;137069;1370...,103921;103922;103923;103924;103925;103926;1039...,103922,NaN,NaN
P04156-2,P04156-2;A2A2V1;P04156,P04156-2;A2A2V1;P04156,1;1;1,1;1;1,1;1;1,Major prion protein,PRNP,>sp|P04156-2|PRIO_HUMAN Isoform 2 of Major pri...,3,1,...,NaN,4,18253,True,19215,255016;255017;255018;255019;255020;255021;2550...,199375,199375,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
REV__Q9C0B6-2,REV__Q9C0B6-2;REV__Q9C0B6,REV__Q9C0B6-2;REV__Q9C0B6,1;1,1;1,1;1,NaN,NaN,>sp|Q9C0B6-2|FAM5B_HUMAN Isoform 2 of Protein ...,2,1,...,NaN,"1,986",17362,True,18274,242825,190062,190062,NaN,NaN
REV__Q9NQR7,REV__Q9NQR7,REV__Q9NQR7,1,1,1,NaN,NaN,>sp|Q9NQR7|CC177_HUMAN Coiled-coil domain-cont...,1,1,...,NaN,"1,987",4228,True,4421,55922;55923;55924;55925;55926;55927;55928;5592...,42720;42721;42722;42723;42724,42722,NaN,NaN
REV__Q9P2F5-2,REV__Q9P2F5-2;REV__Q9P2F5,REV__Q9P2F5-2;REV__Q9P2F5,1;1,1;1,1;1,NaN,NaN,>sp|Q9P2F5-2|STOX2_HUMAN Isoform 2 of Storkhea...,2,1,...,NaN,"1,988",7838,True,8173,103152;103153;103154,77645,77645,NaN,NaN


Get ride of potential contaminants, reverse (decoys) and identified only by a
modification site
reference:
- [cox-labs.github.io/coxdocs/output_tables.html#protein-groups](https://cox-labs.github.io/coxdocs/output_tables.html#protein-groups)

In [9]:
filters = ["Reverse", "Only identified by site", "Contaminant"]
data[filters].describe()

,Reverse,Only identified by site,Contaminant
count,26,24,84
unique,1,1,1
top,+,+,+
freq,26,24,84


In [10]:
mask = data[filters].isna().all(axis=1)
data = data.loc[mask]
data

,Majority protein IDs,Protein IDs,Peptide counts (all),Peptide counts (razor+unique),Peptide counts (unique),Protein names,Gene names,Fasta headers,Number of proteins,Peptides,...,Contaminant,id,Peptide IDs,Peptide is razor,Mod. peptide IDs,Evidence IDs,MS/MS IDs,Best MS/MS,Oxidation (M) site IDs,Oxidation (M) site positions
first_prot,,,,,,,,,,,,,,,,,,,,,
Q13443,Q13443;C9J6H5;C9JPM3;F8WC54;Q13443-2;A0AVL1,Q13443;C9J6H5;C9JPM3;F8WC54;Q13443-2;A0AVL1,1;1;1;1;1;1,1;1;1;1;1;1,1;1;1;1;1;1,Disintegrin and metalloproteinase domain-conta...,ADAM9,>sp|Q13443|ADAM9_HUMAN Disintegrin and metallo...,6,1,...,NaN,0,2726,True,2849,36123;36124;36125;36126;36127;36128;36129;3613...,27484;27485;27486;27487;27488;27489;27490;2749...,27490,NaN,NaN
Q08209,Q08209;Q08209-2;E7ETC2;A1A441;Q08209-3;P16298;...,Q08209;Q08209-2;E7ETC2;A1A441;Q08209-3;P16298;...,4;4;4;4;4;2;2;2;2;2;2;1;1;1;1;1;1;1;1,4;4;4;4;4;2;2;2;2;2;2;1;1;1;1;1;1;1;1,4;4;4;4;4;2;2;2;2;2;2;1;1;1;1;1;1;1;1,Serine/threonine-protein phosphatase 2B cataly...,PPP3CA;PPP3CB,>sp|Q08209|PP2BA_HUMAN Serine/threonine-protei...,19,4,...,NaN,1,699;6176;15271;16451,True;True;True;True,738;6440;16080;17320,9593;9594;9595;9596;80162;80163;80164;80165;80...,7726;7727;60560;60561;60562;163519;163520;1635...,7727;60562;163524;178974,NaN,NaN
A1L4H1,A1L4H1,A1L4H1,1,1,1,Soluble scavenger receptor cysteine-rich domai...,SSC5D,>sp|A1L4H1|SRCRL_HUMAN Soluble scavenger recep...,1,1,...,NaN,2,12914,True,13624,177453;177454,136581;136582,136581,NaN,NaN
P63092,P63092;P04899;P08754;P63096;P09471;P38405;P190...,P63092;P04899;P08754;P63096;P09471;P38405;P190...,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1,Guanine nucleotide-binding protein G(s) subuni...,GNAS;GNAI2;GNAI3;GNAI1;GNAO1;GNAL;GNAT2;GNAT3;...,>sp|P63092|GNAS2_HUMAN Guanine nucleotide-bind...,22,1,...,NaN,3,10218,True,10674,137064;137065;137066;137067;137068;137069;1370...,103921;103922;103923;103924;103925;103926;1039...,103922,NaN,NaN
P04156-2,P04156-2;A2A2V1;P04156,P04156-2;A2A2V1;P04156,1;1;1,1;1;1,1;1;1,Major prion protein,PRNP,>sp|P04156-2|PRIO_HUMAN Isoform 2 of Major pri...,3,1,...,NaN,4,18253,True,19215,255016;255017;255018;255019;255020;255021;2550...,199375,199375,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9Y6B6,Q9Y6B6;Q9H029;D6RDB2;D6RD69,Q9Y6B6;Q9H029;D6RDB2;D6RD69;D6R9R5,3;2;2;2;1,1;1;1;1;1,1;1;1;1;1,GTP-binding protein SAR1b,SAR1B;DKFZp434B2017,>sp|Q9Y6B6|SAR1B_HUMAN GTP-binding protein SAR...,5,3,...,NaN,"1,960",11056;12671;14121,False;True;False,11564;13369;14873,148661;148662;148663;148664;148665;148666;1486...,112725;112726;112727;112728;112729;134047;148469,112729;134047;148469,NaN,NaN
Q9Y6C2,Q9Y6C2,Q9Y6C2;H0Y7A0,3;1,3;1,3;1,EMILIN-1,EMILIN1,>sp|Q9Y6C2|EMIL1_HUMAN EMILIN-1 OS=Homo sapien...,2,3,...,NaN,"1,961",1197;10660;18081,True;True;True,1266;11154;19032,16443;143347;143348;143349;143350;143351;14335...,12937;109132;197424;197425,12937;109132;197425,NaN,NaN
Q9Y6D6,Q9Y6D6;Q9Y6D5;E5RJN9;E5RHL7;E5RIF2,Q9Y6D6;Q9Y6D5;E5RJN9;E5RHL7;E5RIF2,2;1;1;1;1,2;1;1;1;1,2;1;1;1;1,Brefeldin A-inhibited guanine nucleotide-excha...,ARFGEF1;ARFGEF2,>sp|Q9Y6D6|BIG1_HUMAN Brefeldin A-inhibited gu...,5,2,...,NaN,"1,962",16763;18735,True;True,17643;19717,233551;233552;233553;233554;233555;233556;2335...,182340;204240;204241;204242,182340;204240,NaN,NaN


Then we can filter the columns that contain the string `LFQ intensity`. The sample names
are part of the column names (here: `LFQ intensity {sample_name}`)

In [11]:
stub_intensity = "LFQ intensity"
pgs = data.filter(like=stub_intensity)
pgs

,LFQ intensity BAT_NE1,LFQ intensity BAT_NE2,LFQ intensity BAT_NE3,LFQ intensity BAT_NE4,LFQ intensity BAT_NE5,LFQ intensity BAT_woNE1,LFQ intensity BAT_woNE2,LFQ intensity BAT_woNE3,LFQ intensity BAT_woNE4,LFQ intensity BAT_woNE5,LFQ intensity WAT_NE1,LFQ intensity WAT_NE2,LFQ intensity WAT_NE3,LFQ intensity WAT_NE4,LFQ intensity WAT_NE5,LFQ intensity WAT_woNE1,LFQ intensity WAT_woNE2,LFQ intensity WAT_woNE3,LFQ intensity WAT_woNE4,LFQ intensity WAT_woNE5
first_prot,,,,,,,,,,,,,,,,,,,,
Q13443,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"1,131,200"
Q08209,0,0,"10,454,000",0,"8,308,300",0,0,"7,068,900",0,"5,833,600",0,0,0,0,0,0,"13,239,000",0,0,"10,794,000"
A1L4H1,0,0,0,"3,375,200",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
P63092,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"8,852,600"
P04156-2,0,0,0,0,0,0,0,0,0,0,0,0,0,"8,068,200",0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9Y6B6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"1,630,800",0,0,0,0,0
Q9Y6C2,0,0,0,0,0,0,0,0,0,0,0,0,"4,651,500",0,0,0,0,0,0,0
Q9Y6D6,0,0,"4,003,100",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


The associated metadata for protein groups we will keep for reference:

In [12]:
meta_pgs = data.drop(pgs.columns, axis=1)
meta_pgs

,Majority protein IDs,Protein IDs,Peptide counts (all),Peptide counts (razor+unique),Peptide counts (unique),Protein names,Gene names,Fasta headers,Number of proteins,Peptides,...,Contaminant,id,Peptide IDs,Peptide is razor,Mod. peptide IDs,Evidence IDs,MS/MS IDs,Best MS/MS,Oxidation (M) site IDs,Oxidation (M) site positions
first_prot,,,,,,,,,,,,,,,,,,,,,
Q13443,Q13443;C9J6H5;C9JPM3;F8WC54;Q13443-2;A0AVL1,Q13443;C9J6H5;C9JPM3;F8WC54;Q13443-2;A0AVL1,1;1;1;1;1;1,1;1;1;1;1;1,1;1;1;1;1;1,Disintegrin and metalloproteinase domain-conta...,ADAM9,>sp|Q13443|ADAM9_HUMAN Disintegrin and metallo...,6,1,...,NaN,0,2726,True,2849,36123;36124;36125;36126;36127;36128;36129;3613...,27484;27485;27486;27487;27488;27489;27490;2749...,27490,NaN,NaN
Q08209,Q08209;Q08209-2;E7ETC2;A1A441;Q08209-3;P16298;...,Q08209;Q08209-2;E7ETC2;A1A441;Q08209-3;P16298;...,4;4;4;4;4;2;2;2;2;2;2;1;1;1;1;1;1;1;1,4;4;4;4;4;2;2;2;2;2;2;1;1;1;1;1;1;1;1,4;4;4;4;4;2;2;2;2;2;2;1;1;1;1;1;1;1;1,Serine/threonine-protein phosphatase 2B cataly...,PPP3CA;PPP3CB,>sp|Q08209|PP2BA_HUMAN Serine/threonine-protei...,19,4,...,NaN,1,699;6176;15271;16451,True;True;True;True,738;6440;16080;17320,9593;9594;9595;9596;80162;80163;80164;80165;80...,7726;7727;60560;60561;60562;163519;163520;1635...,7727;60562;163524;178974,NaN,NaN
A1L4H1,A1L4H1,A1L4H1,1,1,1,Soluble scavenger receptor cysteine-rich domai...,SSC5D,>sp|A1L4H1|SRCRL_HUMAN Soluble scavenger recep...,1,1,...,NaN,2,12914,True,13624,177453;177454,136581;136582,136581,NaN,NaN
P63092,P63092;P04899;P08754;P63096;P09471;P38405;P190...,P63092;P04899;P08754;P63096;P09471;P38405;P190...,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1,Guanine nucleotide-binding protein G(s) subuni...,GNAS;GNAI2;GNAI3;GNAI1;GNAO1;GNAL;GNAT2;GNAT3;...,>sp|P63092|GNAS2_HUMAN Guanine nucleotide-bind...,22,1,...,NaN,3,10218,True,10674,137064;137065;137066;137067;137068;137069;1370...,103921;103922;103923;103924;103925;103926;1039...,103922,NaN,NaN
P04156-2,P04156-2;A2A2V1;P04156,P04156-2;A2A2V1;P04156,1;1;1,1;1;1,1;1;1,Major prion protein,PRNP,>sp|P04156-2|PRIO_HUMAN Isoform 2 of Major pri...,3,1,...,NaN,4,18253,True,19215,255016;255017;255018;255019;255020;255021;2550...,199375,199375,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9Y6B6,Q9Y6B6;Q9H029;D6RDB2;D6RD69,Q9Y6B6;Q9H029;D6RDB2;D6RD69;D6R9R5,3;2;2;2;1,1;1;1;1;1,1;1;1;1;1,GTP-binding protein SAR1b,SAR1B;DKFZp434B2017,>sp|Q9Y6B6|SAR1B_HUMAN GTP-binding protein SAR...,5,3,...,NaN,"1,960",11056;12671;14121,False;True;False,11564;13369;14873,148661;148662;148663;148664;148665;148666;1486...,112725;112726;112727;112728;112729;134047;148469,112729;134047;148469,NaN,NaN
Q9Y6C2,Q9Y6C2,Q9Y6C2;H0Y7A0,3;1,3;1,3;1,EMILIN-1,EMILIN1,>sp|Q9Y6C2|EMIL1_HUMAN EMILIN-1 OS=Homo sapien...,2,3,...,NaN,"1,961",1197;10660;18081,True;True;True,1266;11154;19032,16443;143347;143348;143349;143350;143351;14335...,12937;109132;197424;197425,12937;109132;197425,NaN,NaN
Q9Y6D6,Q9Y6D6;Q9Y6D5;E5RJN9;E5RHL7;E5RIF2,Q9Y6D6;Q9Y6D5;E5RJN9;E5RHL7;E5RIF2,2;1;1;1;1,2;1;1;1;1,2;1;1;1;1,Brefeldin A-inhibited guanine nucleotide-excha...,ARFGEF1;ARFGEF2,>sp|Q9Y6D6|BIG1_HUMAN Brefeldin A-inhibited gu...,5,2,...,NaN,"1,962",16763;18735,True;True,17643;19717,233551;233552;233553;233554;233555;233556;2335...,182340;204240;204241;204242,182340;204240,NaN,NaN


No we can get rid of the common part `LFQ intensity` and keep only the sample names

In [13]:
pgs.columns = pgs.columns.str.replace(stub_intensity, "").str.strip()
pgs.columns.name = "sample"
pgs

sample,BAT_NE1,BAT_NE2,BAT_NE3,BAT_NE4,BAT_NE5,BAT_woNE1,BAT_woNE2,BAT_woNE3,BAT_woNE4,BAT_woNE5,WAT_NE1,WAT_NE2,WAT_NE3,WAT_NE4,WAT_NE5,WAT_woNE1,WAT_woNE2,WAT_woNE3,WAT_woNE4,WAT_woNE5
first_prot,,,,,,,,,,,,,,,,,,,,
Q13443,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"1,131,200"
Q08209,0,0,"10,454,000",0,"8,308,300",0,0,"7,068,900",0,"5,833,600",0,0,0,0,0,0,"13,239,000",0,0,"10,794,000"
A1L4H1,0,0,0,"3,375,200",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
P63092,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"8,852,600"
P04156-2,0,0,0,0,0,0,0,0,0,0,0,0,0,"8,068,200",0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9Y6B6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"1,630,800",0,0,0,0,0
Q9Y6C2,0,0,0,0,0,0,0,0,0,0,0,0,"4,651,500",0,0,0,0,0,0,0
Q9Y6D6,0,0,"4,003,100",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Parse metadata from column names
The group could be defined in a sample metadata file, but here we just parse it from the
sample names by omitting the numbers at the end of the sample name.

In [14]:
pgs.columns.str.replace(r"\d", "", regex=True)

Index(['BAT_NE', 'BAT_NE', 'BAT_NE', 'BAT_NE', 'BAT_NE', 'BAT_woNE',
       'BAT_woNE', 'BAT_woNE', 'BAT_woNE', 'BAT_woNE', 'WAT_NE', 'WAT_NE',
       'WAT_NE', 'WAT_NE', 'WAT_NE', 'WAT_woNE', 'WAT_woNE', 'WAT_woNE',
       'WAT_woNE', 'WAT_woNE'],
      dtype='object', name='sample')

We add to the information as a MultiIndex of group and sample name to the columns
(sample metadata).

In [15]:
pgs.columns = pd.MultiIndex.from_arrays(
    [pgs.columns.str.replace(r"\d", "", regex=True), pgs.columns],
    names=["group", pgs.columns.name],
)
pgs

group          BAT_NE                                         BAT_woNE  \
sample        BAT_NE1 BAT_NE2    BAT_NE3   BAT_NE4   BAT_NE5 BAT_woNE1   
first_prot                                                               
Q13443              0       0          0         0         0         0   
Q08209              0       0 10,454,000         0 8,308,300         0   
A1L4H1              0       0          0 3,375,200         0         0   
P63092              0       0          0         0         0         0   
P04156-2            0       0          0         0         0         0   
...               ...     ...        ...       ...       ...       ...   
Q9Y6B6              0       0          0         0         0         0   
Q9Y6C2              0       0          0         0         0         0   
Q9Y6D6              0       0  4,003,100         0         0         0   
Q9Y6G9              0       0          0         0         0         0   
Q9Y6I3     11,134,000       0  5,309,200         0 4,596,700         0   

group                                               WAT_NE                    \
sample     BAT_woNE2 BAT_woNE3 BAT_woNE4 BAT_woNE5 WAT_NE1 WAT_NE2   WAT_NE3   
first_prot                                                                     
Q13443             0         0         0         0       0       0         0   
Q08209             0 7,068,900         0 5,833,600       0       0         0   
A1L4H1             0         0         0         0       0       0         0   
P63092             0         0         0         0       0       0         0   
P04156-2           0         0         0         0       0       0         0   
...              ...       ...       ...       ...     ...     ...       ...   
Q9Y6B6             0         0         0         0       0       0         0   
Q9Y6C2             0         0         0         0       0       0 4,651,500   
Q9Y6D6             0         0         0         0       0       0         0   
Q9Y6G9             0         0         0         0       0       0         0   
Q9Y6I3             0 7,092,600         0         0       0       0 8,926,800   

group                            WAT_woNE                                 \
sample       WAT_NE4    WAT_NE5 WAT_woNE1  WAT_woNE2 WAT_woNE3 WAT_woNE4   
first_prot                                                                 
Q13443             0          0         0          0         0         0   
Q08209             0          0         0 13,239,000         0         0   
A1L4H1             0          0         0          0         0         0   
P63092             0          0         0          0         0         0   
P04156-2   8,068,200          0         0          0         0         0   
...              ...        ...       ...        ...       ...       ...   
Q9Y6B6             0  1,630,800         0          0         0         0   
Q9Y6C2             0          0         0          0         0         0   
Q9Y6D6             0          0         0          0         0         0   
Q9Y6G9             0          0         0  9,301,900         0         0   
Q9Y6I3     9,343,600 15,890,000         0          0         0         0   

group                  
sample      WAT_woNE5  
first_prot             
Q13443      1,131,200  
Q08209     10,794,000  
A1L4H1              0  
P63092      8,852,600  
P04156-2            0  
...               ...  
Q9Y6B6              0  
Q9Y6C2              0  
Q9Y6D6              0  
Q9Y6G9      9,676,800  
Q9Y6I3              0  

[1870 rows x 20 columns]

## Long format and log2 transformation
From here we can stack both levels, name the values intensity. If we reset the index we
get the original CKG format.

In [16]:
pgs = pgs.stack([0, 1], future_stack=True).to_frame("intensity")
pgs

intensity
first_prot group    sample              
Q13443     BAT_NE   BAT_NE1            0
                    BAT_NE2            0
                    BAT_NE3            0
                    BAT_NE4            0
                    BAT_NE5            0
...                                  ...
Q9Y6I3     WAT_woNE WAT_woNE1          0
                    WAT_woNE2          0
                    WAT_woNE3          0
                    WAT_woNE4          0
                    WAT_woNE5          0

[37400 rows x 1 columns]

First we  `log2` transform the data. We first set the zeros to `np.nan` to avoid
`-inf` values.

In [17]:
pgs = np.log2(pgs.replace(0.0, np.nan).dropna())
pgs

intensity
first_prot group    sample              
Q13443     WAT_woNE WAT_woNE5     20.109
Q08209     BAT_NE   BAT_NE3       23.318
                    BAT_NE5       22.986
           BAT_woNE BAT_woNE3     22.753
                    BAT_woNE5     22.476
...                                  ...
Q9Y6I3     BAT_NE   BAT_NE5       22.132
           BAT_woNE BAT_woNE3     22.758
           WAT_NE   WAT_NE3       23.090
                    WAT_NE4       23.156
                    WAT_NE5       23.922

[17489 rows x 1 columns]

Data to be saved in the CKG format: Reset the index.

In [18]:
pgs.reset_index()

,first_prot,group,sample,intensity
0,Q13443,WAT_woNE,WAT_woNE5,20.109
1,Q08209,BAT_NE,BAT_NE3,23.318
2,Q08209,BAT_NE,BAT_NE5,22.986
3,Q08209,BAT_woNE,BAT_woNE3,22.753
4,Q08209,BAT_woNE,BAT_woNE5,22.476
...,...,...,...,...
"17,484",Q9Y6I3,BAT_NE,BAT_NE5,22.132
"17,485",Q9Y6I3,BAT_woNE,BAT_woNE3,22.758
"17,486",Q9Y6I3,WAT_NE,WAT_NE3,23.090
"17,487",Q9Y6I3,WAT_NE,WAT_NE4,23.156


Done.